# SubGen on Colab

Sentence-level `.srt` subtitles from English speech (WhisperX), plus a 1080p subtitle video
for YouTube with your own clips as the background.

1. **Runtime → Change runtime type → L4 GPU** (any NVIDIA GPU works; L4 is fastest here).
2. Run the setup cells, then either the **Drive pipeline** or the **web UI**.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d SubGen ] || git clone https://github.com/enoky/SubGen.git
%cd /content/SubGen
!git pull --ff-only

In [ ]:
%%bash
# Install into Colab's own Python, keeping its preinstalled CUDA build of torch when
# WhisperX accepts it (the repo's lockfile targets Windows CUDA 13.2 wheels).
pip install -q uv
python -c "import torch, torchaudio; print(f'torch=={torch.__version__}'); print(f'torchaudio=={torchaudio.__version__}')" > /tmp/torch-pins.txt
PKGS=("whisperx>=3.7.2" "deepmultilingualpunctuation>=1.0.1" "spacy>=3.8,<3.9" "srt>=3.5.3" "tqdm" "gradio" "pillow"
      "en-core-web-lg @ https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl")
uv pip install --system -q -c /tmp/torch-pins.txt "${PKGS[@]}" \
  || { echo "Colab's torch is too old for WhisperX; letting it upgrade torch."; uv pip install --system -q "${PKGS[@]}"; }

### GPU video check

The video pipeline decodes, crops, scales, overlays and encodes on the GPU when ffmpeg
supports it (NVDEC, `scale_cuda`, `overlay_cuda`, NVENC). If Colab's ffmpeg lacks any of
those, this installs a static ffmpeg build that has them. Either way, the pipeline falls back
to slower paths automatically if something isn't available.

In [ ]:
%%bash
cd /content/SubGen
if python -c "import srt2mp4, sys; sys.exit(0 if srt2mp4.ffmpeg_caps().full_gpu('h264') else 1)"; then
  echo "Colab's ffmpeg supports the full GPU path."
else
  echo "Installing a static ffmpeg build with NVENC/NVDEC and CUDA filters..."
  URL=https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-n7.1-latest-linux64-gpl-7.1.tar.xz
  if curl -fsSL -o /tmp/ffmpeg.tar.xz "$URL"; then
    mkdir -p /opt/ffmpeg && tar -xJf /tmp/ffmpeg.tar.xz -C /opt/ffmpeg --strip-components=1
    ln -sf /opt/ffmpeg/bin/ffmpeg /usr/local/bin/ffmpeg
    ln -sf /opt/ffmpeg/bin/ffprobe /usr/local/bin/ffprobe
  else
    echo "Download failed; keeping Colab's ffmpeg (slower fallback paths will be used)."
  fi
fi
ffmpeg -version | head -1
python -m srt2mp4 --check

### Drive pipeline (no UI)

Put your files in Google Drive like this:

```
MyDrive/subgen/
  narration.wav        narration audio
  clips/*.mp4          background clips, joined in filename order (clip2 before clip10)
  script.txt           optional: original script, used for the exact cue wording
  output/              narration.srt and narration.mp4 are saved here
```

The clips are center-cropped to 16:9 and scaled to 1920x1080; their audio is dropped. If they
run shorter than the narration, the last frame is held; if longer, they are cut. The video is
HEVC (NVENC), sized for YouTube 1080p uploads.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content/SubGen
# Extra options are passed to subgen, e.g. --model large-v3-turbo.
# --reuse-srt re-renders the video from output/narration.srt (e.g. after editing it on Drive).
# --srt-only skips the video.
!python -m drive_pipeline --root /content/drive/MyDrive/subgen

### Optional: web UI

Prints a public `gradio.live` link. To cache models on Drive, put
`/content/drive/MyDrive/subgen-models` in the app's **Model cache directory** setting
(mount Drive above first).

In [ ]:
%cd /content/SubGen
from subgen_gradio import build_app

build_app().queue().launch(share=True, debug=True)